In [49]:

from sysdata.sim.db_futures_sim_data import dbFuturesSimData

data = dbFuturesSimData()
dbPrice = data.get_raw_price("SP500")

from sysdata.sim.csv_futures_sim_data import csvFuturesSimData
simdata = csvFuturesSimData()
csvPrice = simdata.get_raw_price("SP500")
#print(dbPrice)
#print(csvPrice)
last_datetime = csvPrice.index[-1]
print(last_datetime)
dbPrice = dbPrice[dbPrice.index > last_datetime]
merged_price = pd.concat([csvPrice, dbPrice])
print(merged_price)


2024-03-28 23:00:00
index
1982-09-14 23:00:00     682.65
1982-09-15 23:00:00     683.15
1982-09-16 23:00:00     682.50
1982-09-17 23:00:00     681.60
1982-09-20 23:00:00     682.40
                        ...   
2025-04-07 08:00:00    4983.50
2025-04-07 09:00:00    4971.00
2025-04-07 10:00:00    5038.50
2025-04-07 11:00:00    5026.50
2025-04-07 12:00:00    5034.50
Name: price, Length: 41915, dtype: float64


In [35]:
from systems.provided.henrik_system.run_system import futures_system

#from sysdata.sim.csv_futures_sim_data import csvFuturesSimData
#simdata = csvFuturesSimData()
system = futures_system()
weights = system.portfolio.get_instrument_weights()
#print(system.portfolio.get_instrument_diversification_multiplier())
print(weights)
#system.portfolio.get_notional_position("HANGTECH")
#profits = system.accounts.portfolio()

#profits.percent.stats()

2025-04-20 13:47:15 DEBUG config {'type': 'config', 'stage': 'config'} Adding config defaults
2025-04-20 13:47:15 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-04-20 13:47:15 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-04-20 13:47:15 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2025-04-20 13:47:15 INFO base_system {'stage': 'portfolio'} Calculating instrument weights
2025-04-20 13:47:15 DEBUG base_system {'stage': 'portfolio'} Calculating raw instrument weights
2025-04-20 13:47:15 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2025-04-20 13:47:15 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2025-04-20 13:47:15 DEBUG base_system Following instruments are marked as 'bad_markets':  ['BAD_EXAMPLE']
2025-04-

missingData: Data for KOSPI_mini not found! Remove from instrument list, or add to config.ignore_instruments

In [ ]:
system.cache.get_itemnames_for_stage("accounts")
system.accounts.portfolio().sharpe()


In [25]:
#system.cache.pickle("private.rob_system.system.pck")

In [ ]:
profits = system.accounts.portfolio()

profits.percent.stats()

In [27]:
import numpy as np
def check_inf(df):
    df = profits.to_frame()
    for col in df.columns:
        if np.isinf(df[col]).any():
            print(col)
            print(df[np.isinf(df[col])][col])

check_inf(profits)

In [ ]:
curve=  profits.curve()
curve.tail()
curve.head()
system.config.base_currency

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_total_assets(asset_series: pd.Series,
                      title: str = "Total Assets Over Time",
                      ma_days: int = 30,
                      currency: str = "HKD",
                      use_plotly: bool = False):
    """
    资产总额可视化函数（静态或交互式）
    
    Parameters:
    - asset_series: pd.Series，index 为日期，值为资产总额
    - title: 图表标题
    - ma_days: 移动平均窗口天数
    - currency: 币种标注（如 "CNY", "USD"）
    - use_plotly: 是否使用 Plotly 交互图（默认 False）
    """
    if not isinstance(asset_series.index, pd.DatetimeIndex):
        raise ValueError("asset_series 的 index 必须为 DatetimeIndex")

    # 平滑趋势线
    ma_series = asset_series.rolling(window=ma_days).mean()

    if use_plotly:
        import plotly.graph_objects as go

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=asset_series.index, y=asset_series,
                                 mode='lines', name='Total Assets',
                                 line=dict(color='blue')))
        fig.add_trace(go.Scatter(x=ma_series.index, y=ma_series,
                                 mode='lines', name=f'{ma_days}-day MA',
                                 line=dict(color='orange', dash='dash')))
        fig.update_layout(title=title,
                          xaxis_title="Date",
                          yaxis_title=f"Amount ({currency})",
                          template="plotly_white")
        fig.show()

    else:
        plt.style.use("seaborn-v0_8-whitegrid")
        fig, ax = plt.subplots(figsize=(14, 6))

        asset_series.plot(ax=ax, color='navy', linewidth=2, label='Total Assets')
        ma_series.plot(ax=ax, color='orange', linestyle='--', linewidth=1.5, label=f'{ma_days}-day MA')

        # 高点注释
        max_val = asset_series.max()
        max_date = asset_series.idxmax()
        ax.annotate(f"Peak: {max_val:,.0f} {currency}",
                    xy=(max_date, max_val),
                    xytext=(max_date, max_val * 1.05),
                    arrowprops=dict(arrowstyle='->', color='gray'),
                    fontsize=10)

        # 日期格式优化
       
        ax.set_title(title, fontsize=18, fontweight='bold')
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel(f"Amount ({currency})", fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()


In [ ]:
#curve.index = curve.index.strftime('%Y-%m-%d')
percent = profits.percent
plot_total_assets(curve, title="Total Assets Over Time", currency=system.config.base_currency)

In [ ]:
df = system.accounts.pandl_for_instrument("ZAR")
system.data.get_raw_price("ZAR")

system.accounts.pandl_for_instrument_rules("ZAR").to_frame()["2023-01-13":"2023-02-10"]
#df[1:20]
#system.portfolio.get_notional_position("ZAR")
#system.portfolio.get_actual_position("ZAR")
# 分析预测值
#system.cache.clear()
system.rawdata.normalised_price_for_asset_class("ZAR")["2023-01-13":"2023-02-10"]

In [ ]:
instruments = system.get_instrument_list()
for instrument in instruments:
    pandl = system.accounts.pandl_for_instrument(instrument)
    plot_total_assets(pandl.curve(), title=instrument, currency=system.config.base_currency)